# 02.1 — Inverted Index

**Problem it solves:** Given millions of documents and a query, find the relevant ones in milliseconds — without reading every document.

**Core idea:** Instead of doc -> words, build word -> [doc1, doc2, ...]. This inverts the natural direction, enabling fast lookup by term.

**Why it matters:** Every search engine (Google, Elasticsearch, Lucene) is built on an inverted index. Understanding this structure makes you understand all of classical IR.

---

## Part 1: Build an Inverted Index from Scratch

In [ ]:
from collections import defaultdict
import re, math

def simple_tokenize(text: str) -> list[str]:
    return re.findall(r'\b[a-z]+\b', text.lower())

class InvertedIndex:
    """
    Core inverted index:
    - posting list: {term -> [(doc_id, positions), ...]}
    - document store: {doc_id -> original text}
    """
    
    def __init__(self):
        # postings: term -> {doc_id -> [positions]}
        self.postings: dict[str, dict[int, list[int]]] = defaultdict(lambda: defaultdict(list))
        self.docs: dict[int, str] = {}
        self.doc_lengths: dict[int, int] = {}
        self.next_id = 0
    
    def add_document(self, text: str, doc_id: int = None) -> int:
        if doc_id is None:
            doc_id = self.next_id
            self.next_id += 1
        
        tokens = simple_tokenize(text)
        self.docs[doc_id] = text
        self.doc_lengths[doc_id] = len(tokens)
        
        for pos, token in enumerate(tokens):
            self.postings[token][doc_id].append(pos)
        
        return doc_id
    
    def get_postings(self, term: str) -> dict[int, list[int]]:
        """Return {doc_id: [positions]} for term."""
        return dict(self.postings.get(term.lower(), {}))
    
    def df(self, term: str) -> int:
        """Document frequency: how many docs contain term."""
        return len(self.postings.get(term.lower(), {}))
    
    def tf(self, term: str, doc_id: int) -> int:
        """Term frequency: how many times term appears in doc."""
        return len(self.postings.get(term.lower(), {}).get(doc_id, []))
    
    @property
    def n_docs(self) -> int:
        return len(self.docs)
    
    def stats(self):
        print(f"Documents: {self.n_docs}")
        print(f"Unique terms: {len(self.postings)}")
        total_postings = sum(len(v) for v in self.postings.values())
        print(f"Total postings: {total_postings}")
        avg_len = sum(self.doc_lengths.values()) / self.n_docs if self.n_docs else 0
        print(f"Avg doc length: {avg_len:.1f} tokens")

# Build index on sample corpus
corpus = {
    0: "the quick brown fox jumps over the lazy dog",
    1: "the dog barked at the fox near the river",
    2: "a quick brown rabbit ran over the hill",
    3: "lazy foxes sleep under old oak trees",
    4: "the river flows near the old oak tree quickly",
    5: "dogs and foxes rarely get along in the wild",
}

idx = InvertedIndex()
for doc_id, text in corpus.items():
    idx.add_document(text, doc_id)

idx.stats()

# Inspect postings
for term in ['fox', 'quick', 'the', 'nonexistent']:
    postings = idx.get_postings(term)
    print(f"\n'{term}' appears in {len(postings)} docs: {dict(postings)}")

## Part 2: Boolean Retrieval with Posting List Intersection

In [ ]:
class BooleanRetriever:
    """
    Boolean AND, OR, NOT queries using posting list operations.
    Intersection (AND) is the key operation — do it on smallest lists first.
    """
    
    def __init__(self, index: InvertedIndex):
        self.index = index
    
    def _and(self, p1: set, p2: set) -> set:
        """AND = intersection. Classic set intersection or merge of sorted lists."""
        return p1 & p2
    
    def _or(self, p1: set, p2: set) -> set:
        return p1 | p2
    
    def _not(self, p: set) -> set:
        return set(self.index.docs.keys()) - p
    
    def search(self, query: str) -> list[int]:
        """
        Parse and execute a boolean query.
        Simple grammar: terms separated by AND/OR/NOT (uppercase).
        Processes left to right (no precedence for simplicity).
        """
        tokens = query.split()
        result = None
        op = 'AND'
        
        for tok in tokens:
            if tok in ('AND', 'OR', 'NOT'):
                op = tok
                continue
            
            docs = set(self.index.get_postings(tok).keys())
            
            if result is None:
                result = docs
            elif op == 'AND':
                result = self._and(result, docs)
            elif op == 'OR':
                result = self._or(result, docs)
            elif op == 'NOT':
                result = self._and(result, self._not(docs))
            op = 'AND'  # default op between adjacent terms
        
        return sorted(result or [])

retriever = BooleanRetriever(idx)

queries = [
    ('fox AND dog', 'docs with both fox AND dog'),
    ('fox OR rabbit', 'docs with fox OR rabbit'),
    ('fox AND NOT dog', 'docs with fox but not dog'),
    ('quick AND brown', 'docs with quick AND brown'),
    ('oak AND river', 'docs with oak AND river'),
]

for query, desc in queries:
    result_ids = retriever.search(query)
    print(f"\nQuery: {query!r} ({desc})")
    for doc_id in result_ids:
        print(f"  [{doc_id}] {corpus[doc_id]}")
    if not result_ids:
        print("  (no results)")

## Part 3: Phrase Search with Positional Index

In [ ]:
def phrase_search(index: InvertedIndex, phrase: str) -> list[int]:
    """
    Find documents where words appear consecutively (phrase query).
    Uses positional postings: for each doc containing all terms,
    check if positions are consecutive.
    """
    terms = simple_tokenize(phrase)
    if not terms:
        return []
    
    # Start with docs containing first term
    candidate_docs = set(index.get_postings(terms[0]).keys())
    
    # Intersect with docs containing all other terms
    for term in terms[1:]:
        candidate_docs &= set(index.get_postings(term).keys())
    
    results = []
    for doc_id in candidate_docs:
        # For each candidate doc, check if positions are consecutive
        positions_per_term = [
            index.get_postings(t)[doc_id] for t in terms
        ]
        
        # Check if first_term_pos + i == ith_term_pos for some starting pos
        first_positions = positions_per_term[0]
        for start_pos in first_positions:
            if all(
                (start_pos + i) in positions_per_term[i]
                for i in range(1, len(terms))
            ):
                results.append(doc_id)
                break
    
    return sorted(results)

phrases = [
    "quick brown",
    "lazy dog",
    "over the",
    "oak tree",
    "brown dog",   # not a phrase in any doc
]

for phrase in phrases:
    results = phrase_search(idx, phrase)
    print(f"Phrase {phrase!r:20} -> doc IDs {results}")
    for doc_id in results:
        print(f"   [{doc_id}] {corpus[doc_id]}")

## Part 4: Index Size and Compression

Real inverted indexes are huge — Lucene uses delta encoding and variable-byte compression on posting lists.

In [ ]:
import sys

# Analyze posting list characteristics
print("Top 10 terms by document frequency (likely stopwords):")
by_df = sorted(idx.postings.items(), key=lambda x: len(x[1]), reverse=True)[:10]
for term, postings in by_df:
    total_tf = sum(len(p) for p in postings.values())
    print(f"  {term:15} df={len(postings):3}  total_tf={total_tf:4}")

print("\nBottom 10 terms (rare terms — high IDF):")
by_df_asc = sorted(idx.postings.items(), key=lambda x: len(x[1]))[:10]
for term, postings in by_df_asc:
    print(f"  {term:15} df={len(postings):3}")

# Delta encoding: posting list [3, 7, 12, 19] -> [3, 4, 5, 7]
# Smaller numbers compress better
def delta_encode(postings: list[int]) -> list[int]:
    if not postings:
        return []
    sorted_p = sorted(postings)
    return [sorted_p[0]] + [sorted_p[i] - sorted_p[i-1] for i in range(1, len(sorted_p))]

def delta_decode(deltas: list[int]) -> list[int]:
    result = []
    running = 0
    for d in deltas:
        running += d
        result.append(running)
    return result

example_list = [3, 7, 12, 19, 23, 101, 102, 500]
encoded = delta_encode(example_list)
decoded = delta_decode(encoded)
print(f"\nDelta encoding:")
print(f"  Original: {example_list}  (max={max(example_list)})")
print(f"  Encoded:  {encoded}  (max={max(encoded)})")
print(f"  Decoded:  {decoded}")
print(f"  Round-trip correct: {decoded == example_list}")

## Summary

**What it does:** Provides O(term) lookup to find all documents containing a term, with position information for phrase queries.

**Key operations:**
- Posting list intersection (AND) — do smallest-first for efficiency
- Union (OR), difference (NOT)
- Positional intersection for phrase queries

**What breaks it:**
- Boolean retrieval returns results in arbitrary order (no ranking) — this is why TF-IDF/BM25 was invented
- Doesn't handle synonyms, typos, or semantic similarity
- High-frequency terms (stopwords like 'the') have huge posting lists — skip lists or omission
- Doesn't scale to web: Lucene/Elasticsearch shard across many machines

---
**Next:** 02.2 — TF-IDF: adding ranking to the inverted index